# SWaT Experiments

## Load Data

In [1]:
# Load swat data
from bdm.Experiments.data_loader import swat_data

train_df, test_df, signal_variables, control_variables, dict_continuous_variables, dict_discrete_variables = swat_data()
train_df = train_df.drop(["Timestamp", "Normal/Attack"], axis=1)
print(train_df.values.shape, test_df.values.shape)

(99360, 51) (89984, 54)


In [2]:
# Prepare data for RS
import numpy as np

x = train_df.values

train_mean = np.mean(x, axis=0, keepdims=True)
train_std = np.std(x, axis=0, keepdims=True)
train_std[train_std == 0] = 1

x -= train_mean
x /= train_std

x = np.nan_to_num(x, nan=0)

# Used 480 for some reason
x_esn_fit = x[:1200]
x_rhy_fit = x[480-100-200-100:480+200]

## Find hyperparams

In [3]:
# Shared ESN backbone from npj (selected as a plain ESN, c = 0, by one-step error) + link-drive calibration
import time
import pandas as pd
import scipy.sparse as sparse
from tqdm import tqdm
from spike_sharing import SpikingNetwork

best_esn_param = (0.19, 0.1, 0.41)                       # (spectral radius, input weight, leakage), npj
BACKBONE = dict(input_dims=x.shape[1], num_nodes=768, average_degree_nodes=8,
                spectral_radius=best_esn_param[0], input_weight=best_esn_param[1], leakage=best_esn_param[2])
THOMAS_DETUNE_SD = 10.0 * 0.2625                         # g_freq x spatial sd of link drive on Thomas (measured)

def drive_stats(net, X, n_steps=2000, skip=200):
    net.reset_initial_states(1); net.reset_history()
    sds, devs = [], []
    for t in tqdm(range(n_steps), mininterval=30):
        net.advance_nodes(X[:, t], save_history=False)    # c = 0: link drive depends on nodes only
        if t >= skip:
            d = (net.incidence_T @ ((net.node_states + 1) / 2)) / net.incidence_norm
            sds.append(d.std()); devs.append(np.abs(d - 0.5).max())
    return np.mean(sds), np.max(devs)

DRIVE_SD, DRIVE_MAXDEV = drive_stats(SpikingNetwork(**BACKBONE, link_strength_change_ratio=0.0), x.T)
G_STAR = THOMAS_DETUNE_SD / DRIVE_SD                     # g_freq giving Thomas-equivalent detuning
G_MAX = 13.5 / DRIVE_MAXDEV                              # keeps I_ref - g*|drive - 0.5| above the I_c + 1 floor
print(f"SWaT drive sd {DRIVE_SD:.4f} (Thomas 0.2625) | g* = {G_STAR:.1f} | floor-safe g <= {G_MAX:.1f}")

100%|████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1008.95it/s]

SWaT drive sd 0.0524 (Thomas 0.2625) | g* = 50.1 | floor-safe g <= 32.7


In [4]:
def group_matrix(net):
    """(D x L) averaging matrix over links whose row node receives input d (npj per-input groups)."""
    rows, _ = net.link_coords()
    r, c = [], []
    for d in range(net.input_dims):
        idx = np.flatnonzero(np.isin(rows, np.flatnonzero(net.input_weights[:, d])))
        r.extend([d] * len(idx)); c.extend(idx)
    G = sparse.csr_matrix((np.ones(len(r)), (r, c)), shape=(net.input_dims, net.num_links))
    size = np.asarray(G.sum(axis=1)).ravel()
    assert np.all(size > 0), "an input group has no links"
    return sparse.diags(1 / size) @ G

def run_features(net, X, ref=None, ref_window=(1000, 3000), desc=""):
    """Online per-input features (no histories saved).
    spike_R[t, d]   = |mean_{i in G_d} e^{i theta_i}|            (npj per-input R, spiking phases)
    spike_ctx[t, d] = RMS_{i in G_d} |rel_lp_i(t) - ref_i|       (deviation from a normal reference context)
    If ref is None, it is the mean rel_lp over ref_window of this run (earlier steps are filled once it exists)."""
    Gm = group_matrix(net)
    T, D = X.shape[1], net.input_dims
    R, C = np.empty((T, D)), np.full((T, D), np.nan)
    net.reset_initial_states(1); net.reset_history()
    buf = np.empty((ref_window[1], net.num_links), np.complex64) if ref is None else None
    for t in tqdm(range(T), desc=desc, mininterval=30):
        net.advance(X[:, t], save_history=False)
        R[t] = np.abs(Gm @ np.exp(1j * net.link_phase))
        if buf is not None:
            buf[t] = net.rel_lp
            if t == ref_window[1] - 1:
                ref = buf[ref_window[0]:].mean(axis=0)
                C[:t + 1] = np.sqrt((Gm @ (np.abs(buf - ref) ** 2).T).T)
                buf = None
        else:
            C[t] = np.sqrt(Gm @ (np.abs(net.rel_lp - ref) ** 2))
    return R, C, ref

# npj's label-free criterion: contrast before vs after frame 480 of training (x_rhy_fit windows).
# Spiking context needs ~500 steps to settle, so prepend 600 steps from an unrelated normal segment.
FIT = np.concatenate([x[2000:2600], x[80:680]]).T                    # 1200 steps; npj windows = last 400
W1, W2 = slice(800, 1000), slice(1000, 1200)                         # before / after frame 480
G_GRID = list(np.round(np.linspace(G_MAX / 2, G_MAX, 3), 1))      # ~16, 25, 33 -> detuning sd 0.9-1.7 (Thomas 2.6)
KAPPA_GRID = [0.04, 0.063, 0.1]                                      # kappa = K / g_freq (Thomas: 0.063)

def contrast(F):
    a, b = F[W1].mean(axis=1), F[W2].mean(axis=1)                    # mean over inputs, per step
    return np.abs(b.mean() - a.mean()) / (a.std() + b.std() + 1e-12)  # scale-free: params change feature scale

search = []
for g in G_GRID:
    for kap in KAPPA_GRID:
        t0 = time.time()
        net = SpikingNetwork(**BACKBONE, g_freq=g, link_coupling=kap * g)
        R, C, _ = run_features(net, FIT, ref_window=(W1.start, W1.stop), desc=f"g={g} k={kap}")
        search.append(dict(g_freq=g, kappa=kap, ctx_contrast=contrast(C), R_contrast=contrast(R),
                           R_mean=R[W1].mean(), minutes=(time.time() - t0) / 60))
        print(search[-1], flush=True)
search = pd.DataFrame(search); search.to_csv("swat_spike_search.csv", index=False)
print(search.sort_values("ctx_contrast", ascending=False).round(3).to_string(index=False))

g=16.3 k=0.04: 100%|███████████████████████████████████████████████████████████████| 1200/1200 [14:28<00:00,  1.38it/s]

{'g_freq': np.float64(16.3), 'kappa': 0.04, 'ctx_contrast': np.float64(1.2804271252221862), 'R_contrast': np.float64(0.1354555373404052), 'R_mean': np.float64(0.9772033202392044), 'minutes': 14.61216121117274}



g=16.3 k=0.063: 100%|██████████████████████████████████████████████████████████████| 1200/1200 [14:04<00:00,  1.42it/s]

{'g_freq': np.float64(16.3), 'kappa': 0.063, 'ctx_contrast': np.float64(1.2548616489851785), 'R_contrast': np.float64(0.22834840480784477), 'R_mean': np.float64(0.9891481650509355), 'minutes': 14.205603349208832}



g=16.3 k=0.1: 100%|████████████████████████████████████████████████████████████████| 1200/1200 [13:39<00:00,  1.47it/s]

...

{'g_freq': np.float64(32.7), 'kappa': 0.04, 'ctx_contrast': np.float64(1.419827972173665), 'R_contrast': np.float64(0.1289005252737794), 'R_mean': np.float64(0.9299671088441812), 'minutes': 12.888010756174724}



g=32.7 k=0.063: 100%|██████████████████████████████████████████████████████████████| 1200/1200 [12:56<00:00,  1.54it/s]

{'g_freq': np.float64(32.7), 'kappa': 0.063, 'ctx_contrast': np.float64(1.44791341885258), 'R_contrast': np.float64(0.14880332012898959), 'R_mean': np.float64(0.9519722210081736), 'minutes': 13.0630579829216}



g=32.7 k=0.1: 100%|████████████████████████████████████████████████████████████████| 1200/1200 [12:53<00:00,  1.55it/s]

{'g_freq': np.float64(32.7), 'kappa': 0.1, 'ctx_contrast': np.float64(1.4480454561263687), 'R_contrast': np.float64(0.18146595768531656), 'R_mean': np.float64(0.9614502672549693), 'minutes': 13.017732576529186}
 g_freq  kappa  ctx_contrast  R_contrast  R_mean  minutes
   24.5  0.100         1.457       0.125   0.986   12.824
   32.7  0.100         1.448       0.181   0.961   13.018
   32.7  0.063         1.448       0.149   0.952   13.063
   32.7  0.040         1.420       0.129   0.930   12.888
   24.5  0.063         1.390       0.162   0.978   12.765
   24.5  0.040         1.326       0.194   0.963   14.162
   16.3  0.040         1.280       0.135   0.977   14.612
   16.3  0.063         1.255       0.228   0.989   14.206
   16.3  0.100         1.246       0.259   0.995   13.769


In [5]:
best = search.sort_values("ctx_contrast", ascending=False).iloc[0]
SPK_PARAMS = dict(g_freq=float(best.g_freq), link_coupling=float(best.kappa * best.g_freq))   # c = 1.0, tau = 60, omega0 = 0.4
print("selected:", SPK_PARAMS)
mm = lambda F: (F - F.min(0, keepdims=True)) / (F.max(0, keepdims=True) - F.min(0, keepdims=True) + 1e-12)
FEATURE = "ctx"                                          # "ctx" or "R": rerun BDM/NSIBF once per feature

selected: {'g_freq': 24.5, 'link_coupling': 2.45}


## Run data through RS

In [ ]:
net = SpikingNetwork(**BACKBONE, **SPK_PARAMS)
train_R, train_C, REF_CTX = run_features(net, x.T, ref_window=(1000, 3000), desc="train")
np.savez("swat_spike_train.npz", R=train_R, C=train_C, ref=REF_CTX)
train_r = mm(train_C if FEATURE == "ctx" else train_R)
assert train_r.shape[1] == x.shape[1]

train:   0%|                                                                                 | 0/99360 [00:00<?, ?it/s]

In [ ]:
# Load test_df
x = test_df.values[:, 1:-2].astype(float)

x -= train_mean
x /= train_std

x = np.nan_to_num(x, nan=0)
test_x_driftbench = x
x.shape

In [ ]:
net = SpikingNetwork(**BACKBONE, **SPK_PARAMS)           # same seed: same links, so the training ref applies
test_R, test_C, _ = run_features(net, x.T, ref=REF_CTX, desc="test")
np.savez("swat_spike_test.npz", R=test_R, C=test_C)
test_r = mm(test_C if FEATURE == "ctx" else test_R)
assert test_r.shape[1] == x.shape[1]

In [ ]:
# Plot example from test
import matplotlib.pyplot as plt

plt.plot(test_r[919:1119])
plt.show()

## BDM

In [ ]:
# Train bdm
# import keras
# keras.config.set_backend("tensorflow")

import bdm.Algorithm.BDM as new_bdm
import pandas as pd

new_train_df = pd.DataFrame(train_r, columns=train_df.columns).astype(float)

xl = 8
ul = xl * 2
batch_size = 256
bdm = new_bdm.BDM(x_length=xl, u_length=ul)
epochs = 250

x_prev, x_curr, x_next, u_curr, _ = bdm.prepare_data(
    new_train_df, signal_variables=signal_variables,
    control_variables=control_variables, dict_continuous_variables=dict_continuous_variables,
    dict_discrete_variables=dict_discrete_variables, label=None, normalization=None, freq=1
)

x_dim = int(x_curr.shape[1]/xl)
index = np.arange(len(x_prev))
train_pos = index[:(len(x_prev) * 3 // 4)]
val_pos = index[(len(x_prev) * 3 // 4):]

x_train_bdm = [x_prev[train_pos, :], x_curr[train_pos, :], x_next[train_pos, :], u_curr[train_pos, :]]
x_val_bdm = [x_prev[val_pos, :], x_curr[val_pos, :], x_next[val_pos, :], u_curr[val_pos, :]]

bdm.train(x_train_bdm, s_dim=4, s_activation='tanh', batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=2)

In [ ]:
# Test bdm
from bdm.Algorithm.utils import f_search
from sklearn import metrics

bdm.estimate_system(x_val_bdm)

new_test_df = pd.DataFrame(
    np.concatenate([test_r, test_df.values[:, -1:]], axis=1),
    columns=list(test_df.columns[1:-2]) + [test_df.columns[-1]]
).astype(float)

_, x_curr, _, u_curr, labels = bdm.prepare_data(
    new_test_df, signal_variables=signal_variables,
    control_variables=control_variables, dict_continuous_variables=dict_continuous_variables,
    dict_discrete_variables=dict_discrete_variables, label='label', normalization=None, freq=xl
)

test_x_bdm = x_curr; test_u_bdm = u_curr

z_scores_bdm = bdm.test(test_x_bdm, test_u_bdm)
z_scores_bdm = np.nan_to_num(z_scores_bdm)
t, _ = f_search(z_scores_bdm, labels[1:])

print('BDM')
print('best-f1', t[0]) # Before removing > 0.7, this was 0.75
print('precision', t[1])
print('recall', t[2])
print('accuracy', (t[3] + t[4]) / (t[3] + t[4] + t[5] + t[6]))
print('TP', t[3])
print('TN', t[4])
print('FP', t[5])
print('FN', t[6])
print('AUC:', metrics.roc_auc_score(labels[1:], z_scores_bdm))

## NSIBF

In [ ]:
# Setup NSIBF
import os, sys
sys.path.append(os.path.abspath("NSIBF"))

import NSIBF.framework.models as models
from NSIBF.framework.HPOptimizer.Hyperparameter import UniformIntegerHyperparameter,ConstHyperparameter, UniformFloatHyperparameter
from NSIBF.framework.HPOptimizer import HPOptimizers
from NSIBF.framework.preprocessing import normalize_and_encode_signals
from NSIBF.framework.preprocessing.signals import SignalSource, ContinousSignal
from NSIBF.framework.utils.metrics import bf_search
from NSIBF.framework.utils import negative_sampler

signals = []
for c in list(test_df.columns[1:-2]):
    signals.append(ContinousSignal(c, SignalSource.sensor, isInput=True, isOutput=True, min_value=0, max_value=1))

seqL = 12
kf = models.NSIBF(signals, window_length=seqL, input_range=seqL*3)

pos = len(new_train_df)*3//4
nsibf_val_df = new_train_df.loc[pos:,:]
nsibf_val_df = nsibf_val_df.reset_index(drop=True)

nsibf_train_df = new_train_df.loc[:pos,:]
nsibf_train_df = nsibf_train_df.reset_index(drop=True)

nsibf_train_df = normalize_and_encode_signals(nsibf_train_df, signals, scaler='min_max')
train_x,train_u,train_y,_ = kf.extract_data(nsibf_train_df)
x_train = [train_x,train_u]
y_train = [train_x,train_y]

x,u,y = [],[],[]
for i in range(20):
    r = 0.05*i
    negative_df = negative_sampler.apply_negative_samples(nsibf_val_df, signals, sample_ratio=r, sample_delta=0.05)
    negative_df = normalize_and_encode_signals(negative_df,signals,scaler='min_max')
    neg_x,neg_u,_,neg_labels = kf.extract_data(negative_df,purpose='AD',freq=seqL,label='class_label')
    neg_labels = neg_labels.sum(axis=1)
    neg_labels[neg_labels<seqL]=0
    neg_labels[neg_labels==seqL]=1
    x.append(neg_x)
    u.append(neg_u)
    y.append(neg_labels)
x = np.concatenate(x)
u = np.concatenate(u)
y = np.concatenate(y)
print(list(y).count(1),len(y))
x_neg = [x,u]
y_neg = y

In [ ]:
# Train NSIBF

hp_list = []
hp_list.append(UniformIntegerHyperparameter('z_dim',1,75)) 
hp_list.append(UniformIntegerHyperparameter('hnet_hidden_layers',1,3))  
hp_list.append(UniformIntegerHyperparameter('fnet_hidden_layers',1,3))
hp_list.append(UniformIntegerHyperparameter('fnet_hidden_dim',32,256))
hp_list.append(UniformIntegerHyperparameter('uencoding_layers',1,3))
hp_list.append(UniformIntegerHyperparameter('uencoding_dim',32,256))
hp_list.append(UniformFloatHyperparameter('l2',0,0.05))
hp_list.append(ConstHyperparameter('epochs',25))
hp_list.append(ConstHyperparameter('save_best_only',True))
hp_list.append(ConstHyperparameter('validation_split',0.1))
hp_list.append(ConstHyperparameter('batch_size',256*8))
hp_list.append(ConstHyperparameter('verbose',2))

optor = HPOptimizers.RandomizedGS(kf, hp_list,x_train, y_train,x_neg,y_neg)
kf,optHPCfg,bestScore = optor.run(n_searches=10,verbose=1)

In [ ]:
# Test NSIBF

nsibf_val_df = normalize_and_encode_signals(nsibf_val_df, signals, scaler='min_max') 
val_x,val_u,val_y,_ = kf.extract_data(nsibf_val_df)

nsibf_test_df = normalize_and_encode_signals(new_test_df,signals,scaler='min_max')
test_x,test_u,_,labels = kf.extract_data(nsibf_test_df, purpose='AD', freq=seqL,label='label')
labels = labels.sum(axis=1)
labels[labels>0]=1

kf.estimate_noise(val_x,val_u,val_y)

z_scores = kf.score_samples(test_x, test_u,reset_hidden_states=True)
# np.savetxt('../results/SWAT/NSIBF_scores',z_scores)
# z_scores = np.loadtxt('../results/SWAT/NSIBF_scores')
recon_scores,pred_scores = kf.score_samples_via_residual_error(test_x,test_u)

z_scores = np.nan_to_num(z_scores)

In [ ]:
t, th = bf_search(z_scores, labels[1:],start=0,end=np.percentile(z_scores,99.9),step_num=10000,display_freq=50,verbose=False)
print('NSIBF')
print('best-f1', t[0])
print('precision', t[1])
print('recall', t[2])
print('accuracy',(t[3]+t[4])/(t[3]+t[4]+t[5]+t[6]))
print('TP', t[3])
print('TN', t[4])
print('FP', t[5])
print('FN', t[6])
print()
 
t, th = bf_search(recon_scores[1:], labels[1:],start=0,end=np.percentile(recon_scores,99.9),step_num=10000,display_freq=50,verbose=False)
print('NSIBF-RECON')
print('best-f1', t[0])
print('precision', t[1])
print('recall', t[2])
print('accuracy',(t[3]+t[4])/(t[3]+t[4]+t[5]+t[6]))
print('TP', t[3])
print('TN', t[4])
print('FP', t[5])
print('FN', t[6])
print()
 
t, th = bf_search(pred_scores, labels[1:],start=0,end=np.percentile(pred_scores,99.9),step_num=10000,display_freq=50,verbose=False)
print('NSIBF-PRED')
print('best-f1', t[0])
print('precision', t[1])
print('recall', t[2])
print('accuracy',(t[3]+t[4])/(t[3]+t[4]+t[5]+t[6]))
print('TP', t[3])
print('TN', t[4])
print('FP', t[5])
print('FN', t[6])
print()

# del nsibf_test_df, nsibf_train_df, nsibf_val_df, train_x, train_u, train_y, val_x, val_u, val_y, kf, HPOptimizers

In [ ]:
new_train_df.to_csv("SWaT_train.csv")
new_new_test_df = new_test_df.drop("label", axis=1)
new_new_test_df["Normal/Attack"] = test_df["Normal/Attack"]
new_new_test_df.to_csv("SWaT_test.csv")

## Driftbench

In [ ]:
# Test using driftbench
from drift_helpers import added_detectors 
from drift_helpers import added_metrics

from driftbench.drift_detection.detectors import (
    RandomGuessDetector,
    RollingMeanDifferenceDetector,
    RollingMeanStandardDeviationDetector,
    AggregateFeatureAlgorithm,
    SlidingKSWINDetector,
    ClusterDetector,
    AutoencoderDetector,
    MMDDetector,
)
from driftbench.drift_detection.metrics import (
    TemporalAUC,
    SoftTAUC,
    AUC,
)

from river.drift import ADWIN, KSWIN, PageHinkley

detectors = [
    # Detectors from driftbench
    AutoencoderDetector(
        hidden_layers=[150, 120, 80, 50, 20, 10, 8],
        retrain_always=True,
        detector=AggregateFeatureAlgorithm(
            agg_feature_func=np.mean,
            algorithm=SlidingKSWINDetector(window_size=20, stat_size=20, offset=10),
        ),
        num_epochs=100,
        batch_size=200,
        lr=0.001,
    ),
    AutoencoderDetector(
        hidden_layers=[150, 120, 80, 50, 20, 10, 8],
        retrain_always=True,
        detector=MMDDetector(window_size=20, stat_size=20, offset=10),
        num_epochs=100,
        batch_size=200,
        lr=0.001,
    ),
    AutoencoderDetector(
        hidden_layers=[120, 80, 50, 20, 10, 8],
        retrain_always=True,
        detector=AggregateFeatureAlgorithm(
            agg_feature_func=np.mean,
            algorithm=SlidingKSWINDetector(window_size=20, stat_size=20, offset=10),
        ),
        num_epochs=100,
        batch_size=200,
        lr=0.001,
    ),
    AutoencoderDetector(
        hidden_layers=[120, 80, 50, 20, 10, 8],
        retrain_always=True,
        detector=MMDDetector(window_size=20, stat_size=20, offset=10),
        num_epochs=100,
        batch_size=200,
        lr=0.001,
    ),
    MMDDetector(window_size=20, stat_size=20, offset=10),
    MMDDetector(window_size=30, stat_size=30, offset=10),
    MMDDetector(window_size=40, stat_size=40, offset=10),
    MMDDetector(window_size=50, stat_size=50, offset=10),
    RollingMeanDifferenceDetector(window_size=20),
    RollingMeanDifferenceDetector(window_size=30),
    RollingMeanDifferenceDetector(window_size=40),
    RollingMeanDifferenceDetector(window_size=50),
    RollingMeanStandardDeviationDetector(window_size=20),
    RollingMeanStandardDeviationDetector(window_size=30),
    RollingMeanStandardDeviationDetector(window_size=40),
    RollingMeanStandardDeviationDetector(window_size=50),
    ClusterDetector(n_centers=15, method="kmeans"),
    ClusterDetector(n_centers=15, method="gaussian mixture"),
    ClusterDetector(n_centers=10, method="kmeans"),
    ClusterDetector(n_centers=10, method="gaussian mixture"),
    ClusterDetector(n_centers=5, method="kmeans"),
    ClusterDetector(n_centers=5, method="gaussian mixture")
]

f1_score = added_metrics.F1_Score()
metrics = [
    f1_score,
    added_metrics.Precision(f1_metric_obj=f1_score),
    added_metrics.Recall(f1_metric_obj=f1_score),
    added_metrics.Delay(f1_metric_obj=f1_score),
    added_metrics.EarlyDetect(f1_metric_obj=f1_score),
    added_metrics.Accuracy(f1_metric_obj=f1_score),
    # TemporalAUC(rule='step'),
    # TemporalAUC(rule='trapez'),
    # SoftTAUC(rule='step'),
    # AUC(),
]

output = []


db_train_y = test_df["label"].values[:(len(test_r)//6)*4]
db_test_y = test_df["label"].values[(len(test_r)//6)*4:]

for detector in tqdm(detectors):
    db_train_pred = detector.predict(test_x_driftbench)
    db_test_pred = db_train_pred[(len(test_r)//6)*4:]
    db_train_pred = db_train_pred[:(len(test_r)//6)*4:]

    db_train_pred_r = detector.predict(test_r)
    db_test_pred_r = db_train_pred_r[(len(test_r)//6)*4:]
    db_train_pred_r = db_train_pred_r[:(len(test_r)//6)*4]

    try:
        f1_score(db_train_pred, db_train_y)
        pred_thresh = f1_score.thresh
        f1_score(db_train_pred_r, db_train_y)
        pred_r_thresh = f1_score.thresh
        
        try:
            output.append({
                "sample": i,
                "detector": detector.name,
                "params": detector.get_hparams(),
                "generic_metrics": {
                    metric.name: metric(db_test_pred, db_test_y, set_threshold=pred_thresh) for metric in metrics
                },
                "r_metrics": {
                    metric.name: metric(db_test_pred_r, db_test_y, set_threshold=pred_r_thresh) for metric in metrics
                }
            })
        except TypeError:
            output.append({
                "sample": i,
                "detector": detector.name,
                "params": detector.get_hparams(),
                "generic_metrics": {
                    metric.name: metric(db_test_pred, db_test_y) for metric in metrics
                },
                "r_metrics": {
                    metric.name: metric(db_test_pred_r, db_test_y) for metric in metrics
                }
            })
    except ValueError:
        pass
    except RuntimeError:
        print("RE", detector.name)
    

In [ ]:
# Separate f1 scores by detector and whether using RS
separated_output = {}
for result in output:
    params = ",".join("%s:%s" % (k, v) for k, v in result["params"].items())
    separated_output[result["detector"] + "," + params] = separated_output.get(result["detector"] + "," + params, {
        "generic": [],
        "rs": []
    })

    separated_output[result["detector"] + "," + params]["generic"].append(result["generic_metrics"])
    separated_output[result["detector"] + "," + params]["rs"].append(result["r_metrics"])

print(separated_output[result["detector"] + "," + params]["rs"][0].keys())

for k, v in separated_output.items():
    print(k, np.nanmean([i["EarlyDetect"] for i in v["generic"]]), np.nanmean([i["EarlyDetect"] for i in v["rs"]]))
    print(k, np.nanmean([i["Delay"] for i in v["generic"]]), np.nanmean([i["Delay"] for i in v["rs"]]))
    print(k, np.nanmean([i["F1_Score"] for i in v["generic"]]), np.nanmean([i["F1_Score"] for i in v["rs"]]))
    print(k, np.nanmean([i["Precision"] for i in v["generic"]]), np.nanmean([i["Precision"] for i in v["rs"]]))
    print(k, np.nanmean([i["Recall"] for i in v["generic"]]), np.nanmean([i["Recall"] for i in v["rs"]]))
    print(k, np.nanmean([i["Accuracy"] for i in v["generic"]]), np.nanmean([i["Accuracy"] for i in v["rs"]]))
    print("======")

In [ ]:
output